# How do we know it's working?

Notebook 04 produced a ranked list. This notebook describes how Sales would use it and how we would **measure whether it helps**.

**Flow:** 02 what makes an attractive merchant → 03 where to find more → 04 who to contact first → **05 how we measure it**

The short version:
1. **Pick one number that defines success:** funded dollars per leads worked.
2. **Track the funnel behind it, tier by tier,** so we can see where each tier wins or loses.
3. **Compare tiers with each other** and reorder them based on results.
4. **Review monthly** and change the list.

## 1. What success looks like

**Primary metric: funded dollars per leads worked, within 90 days of first contact.**

A *lead worked* is a lead a rep has contacted at least once (a logged call, email or meeting). I divide by worked leads, not by every lead on the list, so a tier isn't penalized just because reps didn't get to it. The 90-day window is a placeholder. It should match how long a typical deal takes, which I can't see in this data. The data given has no activity log, so this would need Salesforce activity records or a "date first contacted" field.

**The funnel behind it** follows the stages the Snap data already uses:

| Stage | Metric | How it's calculated | Where it comes from |
|---|---|---|---|
| Reach | Contact rate | leads worked ÷ leads on the list | Salesforce activity |
| Interest | Lead to application | leads with an application ÷ leads worked | Snap applications, matched to the lead |
| Quality | Application to approval | approvals ÷ applications | Snap (`aprv_ytd`, `apps_ytd`) |
| Close | Approval to completion | completions ÷ approvals | Snap (`cmpl_ytd`, `aprv_ytd`) |
| Value | Funded dollars per leads worked | funded $ ÷ leads worked | Snap(`fund_ytd`), Salesforce activity |

**Data-health checks** (the list is only as good as the matching behind it):

| Metric | Why it matters |
|---|---|
| Match rate between datasets | If it drops, the list may start including existing customers |
| Share of records that can't be linked | Shows when a form or feed has broken |
| Duplicate rate per source | A rising rate means records are being double-counted |
| Freshness of the card data | Old activity means businesses may have closed |
| Share of list rows Sales flags as "already a customer" | Direct signal that the exclusions are missing something |

## 2. Where we start: baseline numbers from the data we have

Before changing anything, record what the funnel looks like today, so a later result has something to be compared with. The Snap KPIs are simulated for this exercise, so treat them as illustrative.

In [19]:
from pathlib import Path

import duckdb
import pandas as pd

here = Path.cwd().resolve()
repo_root = next(p for p in [here, *here.parents] if (p / "warehouse" / "snap-assessment.duckdb").exists())
con = duckdb.connect(str(repo_root / "warehouse" / "snap-assessment.duckdb"), read_only=True)

def q(sql: str) -> pd.DataFrame:
    """Run a SQL query and return the results as a Pandas DataFrame."""
    return con.sql(sql).df()

**Snap funnel for active merchants today:**

In [20]:
q("""
select product_type,
       count(*) as active_merchants,
       round(sum(apps_ytd) / count(*), 1) as apps_per_merchant,
       round(100.0 * sum(aprv_ytd) / nullif(sum(apps_ytd), 0), 1) as approval_rate_pct,
       round(100.0 * sum(cmpl_ytd) / nullif(sum(apps_ytd), 0), 1) as completion_rate_pct,
       round(sum(fund_ytd) / count(*)) as funded_per_merchant
from "snap-assessment".staging.stg_snap
where not is_inactivated and dupe_record_rank = 1
group by product_type
""")

,product_type,active_merchants,apps_per_merchant,approval_rate_pct,completion_rate_pct,funded_per_merchant
0,Lease,425,82.1,60.3,39.9,48547.0
1,Loan,229,28.2,51.5,34.6,15433.0


**Salesforce today:** how leads are spread across statuses. It is the closest thing to a lead-to-customer rate in the data, but it depends on what `Converted` means, which is still an open question (see 03).

In [21]:
q("""
select status,
       count(*) as leads,
       round(100.0 * count(*) / sum(count(*)) over (), 1) as pct_of_leads
from "snap-assessment".staging.stg_salesforce_list
where dupe_record_rank = 1
group by status
order by leads desc
""")

,status,leads,pct_of_leads
0,Disqualified,616,48.5
1,Converted,466,36.7
2,Nurturing,173,13.6
3,Working,5,0.4
4,Merchant Invitation Sent,4,0.3
5,MQL,3,0.2
6,BDR Working,1,0.1
7,Awaiting Approval,1,0.1
8,New,1,0.1
9,Submit to Join,1,0.1


What the baseline *can't* tell me: how long a deal takes, what a rep's capacity is, and what a cold lead converts at, because the card and purchased lists have no outcomes. The first weeks of tracking would fill those gaps.

## 3. How we'd read the results: compare the tiers

Every business in notebook 04 has a tier. After Sales works the list for a period, I would report the same funnel **for each tier** and compare.

| Tier | Play | Businesses |
|---|---|---|
| 1 | Inbound | 475 |
| 2 to 4 | Pipeline | 16, 171 and 40 |
| 5 | Win-back | 506 |
| 6 to 8 | New prospecting | 122, 698 and 514 |

What I'd look for:
- **Does warm beat cold?** The order of the tiers is my hypothesis (inbound, then pipeline, then win-back, then new prospecting). If the funded dollars per leads worked don't follow that order, the order should change.
- **Which cold tier is worth the effort?** Compare tiers 6, 7 and 8. This tells us whether two lists agreeing (tier 6) is a better signal than one list alone.
- **Small tiers are anecdotes.** Tiers 2 and 4 have 16 and 40 businesses, so I would just work them all and not draw conclusions from their rates.

**What this can't prove:** if results improve, the list isn't necessarily the cause (reps may work harder, or the market may move). A holdout test, where some leads in each tier are handled the way Sales handles them today, would answer that. I would add it once Sales has agreed to run one.

## 4. How Sales would actually use this

| Step | What happens | Owner |
|---|---|---|
| Refresh | The list is rebuilt on a schedule (weekly to start) from the five sources | Data / analytics |
| Deliver | Each business gets a `tier` and `priority_rank` in Salesforce, so reps see the order in the tool they already use | Sales operations |
| Work | Reps work the list top down, within their capacity | Sales |
| Record outcomes | Reps log a reason when they skip or reject a lead ("already a customer," "closed," "wrong contact") | Sales |
| Review | Weekly funnel by tier, monthly decision on changes | Sales leadership + analytics |

The **feedback loop** matters most. Reps' rejection reasons tell us where the matching or exclusions are wrong, which makes the next list better. In production, the tier rules from notebook 04 would become a tested dbt model, so the list refreshes without anyone rerunning notebooks.

**Before launch, check with compliance** how the purchased and card-derived lists may be used for outreach. Contacting businesses from purchased data can have legal rules that I haven't assessed.

## 5. What would change the plan

| If we see… | Then… |
|---|---|
| A tier's funded $ per 100 leads worked is high and covers the cost of working it | Give it more rep time |
| A tier is low | Move it lower, or stop working it |
| Two tiers are ordered differently from my assumption | Reorder by results |
| Approval rate falls in a tier | Tighten the filter |
| Skip reasons are mostly "already a customer" | Fix the matching before anything else |
| Match rate or link rate drops | Check the data feed first. Don't trust the list until it's fixed |

## 6. Risks and limits

- **No outcome data for card and purchased lists.** Lookalike fit is an assumption until this is measured. The tiers are hypotheses.
- **The Snap KPIs are simulated.** Real benchmarks may differ.
- **Exact matching.** Some businesses will be missed or double-counted, so counts are approximate.
- **Capacity.** Sales can only work so many. The first few hundred matter most, so the order within and across tiers is where most of the value is.
- **Win-back is untested.** I don't know why those merchants went inactive.

## 7. What I would do next with more time

1. **Better matching.** Fuzzy name matching and cleaner street suffixes, then re-measure how many businesses change tiers.
2. **Confirm what `Converted` means**, and add the cleared leads back to the right tier.
3. **Find out why merchants went inactive**, to decide whether win-back is worth a tier.
4. **Add a holdout test** to show the list causes the improvement.
5. **Explore location and rep capacity**, so the list is also organized by territory.

## Takeaways

1. **The one number:** funded dollars per leads worked, within 90 days of first contact.
2. **How I'd read it:** the same funnel for each tier, compared with each other, reviewed monthly.
3. **What it can't prove yet:** that the list caused the improvement. That needs a holdout test.

## close connection

In [22]:
con.close()